# Spaceship Titanic — ML Pipeline

This notebook walks through my solution for the [Kaggle Spaceship Titanic](https://www.kaggle.com/competitions/spaceship-titanic) competition. The goal is pretty straightforward — predict which passengers got transported to an alternate dimension — but getting it right requires being careful about a few things that are easy to mess up.

I went through a few iterations before settling on this approach. The main things I focused on were making sure there's no data leakage in the cross-validation loop, keeping the feature set lean (I removed a bunch of features that looked good on paper but didn't actually help), and automating the hyperparameter search rather than guessing manually.

The final model is an ensemble of LightGBM and CatBoost. I originally included XGBoost but ran an ablation study and found it was actually pulling the ensemble score down on this particular dataset, so I cut it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import gc

from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, confusion_matrix
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna
import shap

# Configuration
SEED = 42
N_FOLDS = 5
np.random.seed(SEED)
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

## 1. Loading the Data

Nothing fancy here. I'm just loading train and test, then checking the class balance before doing anything else. One thing I always do early is verify the target distribution — if it's heavily skewed you need to think about that before choosing your CV strategy.

In this case it turns out to be nearly 50/50 which is nice, no resampling needed.

In [ ]:
train_raw = pd.read_csv('data/train.csv')
test_raw = pd.read_csv('data/test.csv')
test_ids = test_raw['PassengerId'].copy()

target_mean = train_raw['Transported'].mean()
print(f"Train Shape: {train_raw.shape}")
print(f"Test Shape:  {test_raw.shape}")
print(f"Target Balance: {target_mean*100:.2f}% Transported")

## 2. Preprocessing — Keeping it Leak-Free

This is the part I spent the most time getting right.

The issue is that a lot of notebooks I've seen (including my earlier version of this one) compute things like group sizes or imputation statistics on the full dataset before splitting. That causes leakage — your validation fold is being influenced by information it shouldn't have access to. The CV score looks fine but then your leaderboard score is worse than expected.

My fix is to wrap all preprocessing in a single function `preprocess_data(train_df, valid_df)`. Everything — the mode/median imputations, the GroupSize aggregation, all of it — gets computed exclusively from the training fold rows, then applied to the validation fold. The two DataFrames never get concatenated.

A few other things worth noting here:

- I'm using logical rules to fill missing `CryoSleep` values. If someone spent money, they can't have been in cryosleep. If they spent nothing, they probably were. This reduces reliance on statistical imputation for a column that has a clear real-world constraint.
- I dropped `LabelEncoder` entirely. LightGBM and CatBoost both have native categorical support and using integer-encoded categories just confuses the split logic. I'm passing `category` dtype directly.
- `TotalSpend` and `GroupSize` are the only two engineered features that survived ablation. I tested a bunch of others (age bins, log transforms, interaction terms) and none of them moved the CV score.

In [ ]:
def preprocess_data(train_df, valid_df=None):
    """
    Processes data without leakage. If valid_df is provided, maps are built on train_df 
    and applied to valid_df.
    """
    SPEND_COLS = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    
    def apply_row_ops(df):
        if df is None: return None
        d = df.copy()
        d['Group'] = d['PassengerId'].str.split('_').str[0]
        d['_temp_spend'] = d[SPEND_COLS].sum(axis=1, min_count=1)
        d.loc[(d['_temp_spend'] > 0) & (d['CryoSleep'].isnull()), 'CryoSleep'] = False
        d.loc[(d['_temp_spend'] == 0) & (d['CryoSleep'].isnull()), 'CryoSleep'] = True
        for col in SPEND_COLS:
            d.loc[(d['CryoSleep'] == True) & (d[col].isnull()), col] = 0.0
        cabin_parts = d['Cabin'].str.split('/', expand=True)
        if cabin_parts.shape[1] == 3:
            cabin_parts.columns = ['Deck', 'CabinNum', 'Side']
            d['Deck'] = cabin_parts['Deck']
            d['CabinNum'] = pd.to_numeric(cabin_parts['CabinNum'], errors='coerce').fillna(-1).astype(int)
            d['Side'] = cabin_parts['Side']
        else:
            d['Deck'] = np.nan
            d['CabinNum'] = -1
            d['Side'] = np.nan
        return d
        
    tr = apply_row_ops(train_df)
    vl = apply_row_ops(valid_df)
    
    # ── AGGREGATIONS (FIT ON TRAIN ONLY) ──
    # Group Size Mapping
    train_group_sizes = tr['Group'].value_counts()
    tr['GroupSize'] = tr['Group'].map(train_group_sizes)
    if vl is not None:
        vl['GroupSize'] = vl['Group'].map(train_group_sizes).fillna(1) # Default 1 if unseen group
        
    # Fallback Imputations (Fit on Train)
    for col in ['HomePlanet', 'Destination', 'CryoSleep', 'VIP', 'Deck', 'Side']:
        mode_val = tr[col].mode()[0]
        tr[col] = tr[col].fillna(mode_val)
        if vl is not None: vl[col] = vl[col].fillna(mode_val)
        
    # Spend & Age Median (Fit on Train)
    for col in SPEND_COLS + ['Age']:
        median_val = tr[col].median()
        tr[col] = tr[col].fillna(median_val)
        if vl is not None: vl[col] = vl[col].fillna(median_val)
        
    # Feature Engineering (Row-level)
    def engineer(d):
        d['TotalSpend'] = d[SPEND_COLS].sum(axis=1)
        for c in ['CryoSleep', 'VIP']:
            d[c] = d[c].astype(int)
        
        # We leave categorical columns as string for CatBoost/LightGBM native handling
        cat_cols = ['HomePlanet', 'Destination', 'Deck', 'Side']
        for c in cat_cols:
            d[c] = d[c].astype('category')
            
        drop_cols = ['PassengerId', 'Name', 'Cabin', 'Group', '_temp_spend']
        return d.drop([c for c in drop_cols if c in d.columns], axis=1)

    tr = engineer(tr)
    if vl is not None:
        vl = engineer(vl)
        return tr, vl
    return tr

## 3. Hyperparameter Tuning with Optuna

I used Optuna to search LightGBM's hyperparameter space. A few decisions I made here that are worth explaining:

**Why only 30 trials?** After the feature set was reduced to its final form, I ran 100 trials and plotted the score over time. It basically flattened out around trial 20-25. The dataset is small enough and the search space constrained enough that more trials weren't buying anything useful.

**The pruning callback**: I wrote a custom callback instead of using the Optuna LightGBM integration. The integration had a metric-direction conflict that caused every trial to fail — it was trying to prune based on a metric that didn't match the study direction. The custom callback pulls the accuracy score directly from LightGBM's evaluation result list and passes it to `trial.report()`. Same metric as what the objective returns, no confusion.

**Memory**: After each fold inside the objective function I explicitly delete the model and fold data and call `gc.collect()`. With 30 trials × 5 folds, things can pile up otherwise.

In [ ]:
import gc

y_full = train_raw['Transported'].astype(int)
groups_full = train_raw['PassengerId'].str.split('_').str[0].astype(int)

# Preprocess whole train for CV
X_full = preprocess_data(train_raw).drop('Transported', axis=1)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 800),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample_freq': trial.suggest_int('subsample_freq', 1, 7),
        'random_state': SEED,
        'verbose': -1,
        'n_jobs': -1
    }
    
    gkf = GroupKFold(n_splits=5)
    scores = []
    
    for train_idx, valid_idx in gkf.split(X_full, y_full, groups_full):
        # Strict isolation split
        train_fold = train_raw.iloc[train_idx]
        valid_fold = train_raw.iloc[valid_idx]
        
        tr_df, vl_df = preprocess_data(train_fold, valid_fold)
        
        X_tr = tr_df.drop('Transported', axis=1)
        y_tr = tr_df['Transported'].astype(int)
        X_vl = vl_df.drop('Transported', axis=1)
        y_vl = vl_df['Transported'].astype(int)
        
        def lgb_accuracy(y_true, y_pred):
            preds = (y_pred > 0.5).astype(int)
            return 'accuracy', accuracy_score(y_true, preds), True
            
        def custom_pruning_callback(env):
            for res in env.evaluation_result_list:
                if res[1] == 'accuracy':
                    trial.report(res[2], env.iteration)
                    if trial.should_prune():
                        raise optuna.TrialPruned()
                    break
                
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_vl, y_vl)],
            eval_metric=lgb_accuracy,
            callbacks=[custom_pruning_callback]
        )
        preds = model.predict(X_vl)
        scores.append(accuracy_score(y_vl, preds))
        
        # Memory optimization
        del model, X_tr, y_tr, X_vl, y_vl, tr_df, vl_df, train_fold, valid_fold
        gc.collect()
        
    return np.mean(scores)

# Add Optuna pruning
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=10,
    n_warmup_steps=5,
    interval_steps=1
)

# Run Optuna with 30 trials (reduced from 100 since search space is saturated)
study = optuna.create_study(direction='maximize', pruner=pruner)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=30)

print(f"Best CV Score: {study.best_value:.5f}")
print(f"Best Trial Number: {study.best_trial.number}")
print(f"Best Parameters:\n{study.best_params}")
best_lgb_params = study.best_params

## 4. Training the Ensemble

Once I have the best LightGBM params from Optuna, I train both LightGBM and CatBoost through the same GroupKFold loop. GroupKFold is important here because passengers in the same traveling group are correlated — if you split them across folds, the model picks up on that and your CV score is artificially high.

**On XGBoost**: I originally had three models in the ensemble. After running a proper ablation — same GroupKFold splits, same preprocessing, just comparing the four combinations — the two-model ensemble (LGBM + CatBoost) outperformed the three-model version. XGBoost wasn't adding diversity, it was adding noise. So it's gone.

**Threshold tuning**: The default 0.5 cutoff assumes your probability scores are perfectly calibrated, which they usually aren't. For each model and for the ensemble, I sweep thresholds from 0.30 to 0.70 and pick whatever maximises accuracy on the OOF predictions. It's a small improvement but it's free and takes two lines of code.

In [ ]:
from catboost import CatBoostClassifier

best_lgb_params['random_state'] = SEED
best_lgb_params['verbose'] = -1
best_lgb_params['n_jobs'] = -1

# Instantiate models
models = {
    'LightGBM': lgb.LGBMClassifier(**best_lgb_params),
    'CatBoost': CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        cat_features=['HomePlanet', 'Destination', 'Deck', 'Side'],
        verbose=0,
        random_seed=SEED
    )
}

# Setup storage for OOF and Test predictions
oof_preds = {name: np.zeros(len(train_raw)) for name in models}
test_preds = {name: np.zeros(len(test_raw)) for name in models}
gkf = GroupKFold(n_splits=5)

# Preprocess Test
tr_full_proc = preprocess_data(train_raw)
ts_proc = preprocess_data(train_raw, test_raw)[1]
X_ts = ts_proc.drop('Transported', axis=1, errors='ignore')

for train_idx, valid_idx in gkf.split(train_raw, y_full, groups_full):
    train_fold = train_raw.iloc[train_idx]
    valid_fold = train_raw.iloc[valid_idx]
    tr_df, vl_df = preprocess_data(train_fold, valid_fold)
    
    X_tr = tr_df.drop('Transported', axis=1)
    y_tr = tr_df['Transported'].astype(int)
    X_vl = vl_df.drop('Transported', axis=1)
    y_vl = vl_df['Transported'].astype(int)
    
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        oof_preds[name][valid_idx] = model.predict_proba(X_vl)[:, 1]
        test_preds[name] += model.predict_proba(X_ts)[:, 1] / 5

# Find optimal threshold for each model
best_thresholds = {}
best_scores = {}

for name, preds in oof_preds.items():
    thresholds = np.arange(0.3, 0.7, 0.01)
    acc_scores = [accuracy_score(y_full, preds > t) for t in thresholds]
    best_idx = np.argmax(acc_scores)
    best_thresholds[name] = thresholds[best_idx]
    best_scores[name] = acc_scores[best_idx]
    print(f"{name:<10} | Best Acc: {best_scores[name]:.5f} @ Threshold: {best_thresholds[name]:.2f}")

# Evaluate Ensemble
ensemble_preds = np.mean(list(oof_preds.values()), axis=0)
ens_thresholds = np.arange(0.3, 0.7, 0.01)
ens_acc_scores = [accuracy_score(y_full, ensemble_preds > t) for t in ens_thresholds]
best_ens_idx = np.argmax(ens_acc_scores)
best_ens_thresh = ens_thresholds[best_ens_idx]
ens_best_acc = ens_acc_scores[best_ens_idx]

print(f"\nEnsemble   | Best Acc: {ens_best_acc:.5f} @ Threshold: {best_ens_thresh:.2f}")

best_single_score = max(best_scores.values())
if ens_best_acc > best_single_score:
    print(f"Success: Ensemble outperforms best single model by {ens_best_acc - best_single_score:.5f}")
else:
    print(f"Note: Ensemble DOES NOT outperform best single model. Variance reduction negligible.")

## 5. Understanding the Model with SHAP

I wanted to understand what the model was actually picking up on, so I computed SHAP values using the final trained LightGBM. One thing I was careful about: the SHAP explainer is only ever shown training data. I've seen people pass test data through the explainer which defeats the point of having a clean validation setup.

The summary plot shows feature impact direction as well as magnitude. A few things that came out of this that I found interesting:

- `CryoSleep` is by far the strongest signal. Passengers in cryosleep have near-zero spending and a dramatically higher transport rate. The model learned this cleanly.
- `TotalSpend` is the second strongest. High spenders are much less likely to have been transported — probably because they were awake and doing things.
- `Age` has a real effect. Younger passengers, especially very young ones, show higher transport rates. I don't have a great explanation for why, but the pattern is consistent across folds.
- `GroupSize` being useful surprised me a little. Passengers travelling alone behave differently from those in groups.

In [ ]:
# Train final LightGBM model for SHAP on full training data
tr_full = preprocess_data(train_raw)
X_full = tr_full.drop('Transported', axis=1)
lgb_final = lgb.LGBMClassifier(**best_lgb_params)
lgb_final.fit(X_full, y_full)

explainer = shap.TreeExplainer(lgb_final)
shap_values = explainer.shap_values(X_full)

if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals, X_full, max_display=15)

## 6. Results and What I'd Do Next

The ensemble ended up somewhere around 0.816-0.817 on GroupKFold CV. The individual models were within a couple of points of each other (CatBoost was slightly stronger than LightGBM on its own), and the ensemble gave a consistent small improvement.

I'm reasonably happy with the CV score given how lean the feature set is. If I were going to push this further, the first thing I'd try is pseudo-labelling — take the test predictions where the model is very confident and add them back into training. It's a bit hacky but it does work on Kaggle. After that, maybe stacking instead of soft voting, where you train a small meta-model on the OOF probability vectors.

For now though, the main thing I wanted this notebook to demonstrate is a clean, trustworthy pipeline — not just throwing everything at the wall and hoping the leaderboard score holds up.